# 🎰 AI 로또 6/45 번호 분석기 & 스마트 조합 생성기 (v2.0)

> **주요 업그레이드 내역**:
- 🚀 **엑셀 업로드 불필요**: 로컬 엑셀(`lotto_history.xlsx`) 자동 인식 + 구글 코랩 실행 시 동행복권 공식 API로 최신 회차(1회~현재) 1초 자동 수집!
- 📊 **고급 통계 지표 추가**: 산술적 복합도(AC값), 끝수 분석, 구간별(5개 색상대) 분포, 장기 미출현(콜드수) & 최근 대세(핫수) 추적
- 🎯 **5가지 전략 조합 생성기**: 황금 밸런스 AI, 콜드수 반등형, 핫수 추종형, 고정수(반자동), 제외수 필터
- 🔍 **과거 당첨 백테스팅 시뮬레이터**: 내 번호가 역대 전체 회차에서 몇 등까지 당첨되었었는지 즉시 시뮬레이션
- 🎨 **실제 로또 공 색상(5색) 시각화 및 스타일링**

In [ ]:
# ==========================================
# 1. 필수 라이브러리 및 데이터 자동 로드
# ==========================================
import os
import json
import time
import random
from itertools import combinations
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import urllib.request
import ssl

# 엑셀 파일이 없으면 동행복권 API에서 역대 전회차 자동 수집
excel_file = "lotto_history.xlsx"
if not os.path.exists(excel_file):
    print("⚡ 로컬 엑셀 파일이 없어 동행복권 공식 서버에서 역대 전체 회차를 1초 만에 자동 수집합니다...")
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    
    ts = int(time.time() * 1000)
    url = f"https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do?srchLtEpsd=all&_={ts}"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, context=ctx) as resp:
        raw_data = json.loads(resp.read().decode("utf-8"))
        item_list = raw_data.get("data", {}).get("list", [])
        
    rows = []
    for item in item_list:
        nums = sorted([item["tm1WnNo"], item["tm2WnNo"], item["tm3WnNo"], item["tm4WnNo"], item["tm5WnNo"], item["tm6WnNo"]])
        rows.append({
            "회차": item["ltEpsd"],
            "추첨일": item.get("ltRflYmd", ""),
            1: nums[0], 2: nums[1], 3: nums[2], 4: nums[3], 5: nums[4], 6: nums[5],
            "보너스": item["bnsWnNo"],
            "1등당첨자수": item.get("rnk1WnNope", 0),
            "1등당첨금액": item.get("rnk1WnAmt", 0)
        })
    df = pd.DataFrame(rows).sort_values("회차")
    df.to_excel(excel_file, index=False)
    print(f"✅ {len(df)}개 회차 수집 완료 및 {excel_file} 저장!")
else:
    df = pd.read_excel(excel_file)
    print(f"✅ 로컬 파일({excel_file})에서 총 {len(df)}개 회차 데이터를 성공적으로 불러왔습니다!")

# 번호 컬럼 추출 (1~6번 공 및 보너스)
lotto_cols = [1, 2, 3, 4, 5, 6]
lotto_nums = df[lotto_cols]
latest = df.iloc[-1]
bonus_val = latest.get('보너스', latest.get('보너스번호', ''))
print(f"📌 최신 회차: {latest['회차']}회 | 당첨번호: {list(latest[lotto_cols])} + 보너스: {bonus_val}")


## 2. 역대 통계 심층 분석
- 번호별 빈도수 (Top 10 / Bottom 10)
- 동반 출현 2개(짝꿍수) & 3개 조합
- 장기 미출현(콜드수) 순위
- 총합 분포, 홀짝 및 고저 비율

In [ ]:
# ==========================================
# 2. 통계 지표 계산
# ==========================================
all_numbers = lotto_nums.values.flatten()
count_series = pd.Series(all_numbers).value_counts().sort_index()

print("="*50)
print("🔥 [가장 많이 나온 번호 Top 10]")
top10 = count_series.sort_values(ascending=False).head(10)
for rank, (num, cnt) in enumerate(top10.items(), 1):
    print(f"{rank:2d}위: {num:2d}번 ({cnt}회 출현, {cnt/len(df)*100:.1f}%)")

print("-"*30)
print("❄️ [가장 적게 나온 번호 Bottom 10]")
bottom10 = count_series.sort_values(ascending=True).head(10)
for rank, (num, cnt) in enumerate(bottom10.items(), 1):
    print(f"{rank:2d}위: {num:2d}번 ({cnt}회 출현, {cnt/len(df)*100:.1f}%)")

print("-"*30)
# 동반 출현 2개 조합 Top 10
pair_counts = Counter()
for _, row in lotto_nums.iterrows():
    pair_counts.update(combinations(sorted(row.values), 2))
print("🤝 [가장 많이 함께 나온 2개 조합 Top 10]")
for rank, (pair, cnt) in enumerate(pair_counts.most_common(10), 1):
    print(f"{rank:2d}위: {pair} -> {cnt}회 동반 출현")

print("-"*30)
# 미출현 기간 (장기 미출현 콜드수)
latest_rnd = df["회차"].max()
last_seen = {i: 0 for i in range(1, 46)}
for _, row in df.iterrows():
    r = row["회차"]
    for n in row[lotto_cols]:
        last_seen[n] = max(last_seen[n], r)
cold_ranking = sorted([(n, latest_rnd - last_seen[n]) for n in range(1, 46)], key=lambda x: x[1], reverse=True)

print("🧊 [장기 미출현 번호(콜드수) Top 10]")
for rank, (num, gap) in enumerate(cold_ranking[:10], 1):
    print(f"{rank:2d}위: {num:2d}번 ({gap}회차 연속 미출현, 마지막: {last_seen[num]}회)")
print("="*50)


## 3. 시각화 대시보드
- 로또 용지 형태 5행 9열 히트맵
- Top 15 / Bottom 15 차트
- 번호 총합 정규분포 히스토그램 & 평균
- 홀짝 및 고저 비율 파이 차트

In [ ]:
# ==========================================
# 3. 데이터 시각화
# ==========================================
# 한글 폰트 설정 (Windows / Colab 자동 감지)
import platform
if platform.system() == "Windows":
    plt.rc("font", family="Malgun Gothic")
else:
    !sudo apt-get install -y fonts-nanum > /dev/null 2>&1
    plt.rc("font", family="NanumBarunGothic")
plt.rcParams["axes.unicode_minus"] = False

# 1. 로또 용지 형태 히트맵 (5행 9열)
lotto_grid = np.zeros((5, 9))
for num in range(1, 46):
    row = (num - 1) // 9
    col = (num - 1) % 9
    lotto_grid[row, col] = count_series.get(num, 0)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (1) 히트맵
sns.heatmap(lotto_grid, annot=True, fmt=".0f", cmap="YlGnBu", ax=axes[0, 0],
            xticklabels=[str(i) for i in range(1, 10)],
            yticklabels=[f"{i*9+1}~{i*9+9}" for i in range(5)])
axes[0, 0].set_title("로또 번호별 출현 빈도 (용지 히트맵)", fontsize=14, fontweight="bold")

# (2) Top 10 & Bottom 10 막대그래프
top_bot = pd.concat([top10.head(8), bottom10.head(8)])
colors = ["#e74c3c"]*8 + ["#2b8be8"]*8
axes[0, 1].bar([str(i) for i in top_bot.index], top_bot.values, color=colors)
axes[0, 1].set_title("최다 출현 8개(빨강) vs 최소 출현 8개(파랑)", fontsize=14, fontweight="bold")
axes[0, 1].set_ylabel("출현 횟수")

# (3) 번호 총합 분포
sums = lotto_nums.sum(axis=1)
mean_s = sums.mean()
std_s = sums.std()
sns.histplot(sums, bins=30, kde=True, color="#8e44ad", ax=axes[1, 0])
axes[1, 0].axvline(mean_s, color="red", linestyle="--", label=f"평균: {mean_s:.1f}")
axes[1, 0].axvline(mean_s - std_s, color="green", linestyle=":", label=f"추천구간({mean_s-std_s:.0f}~{mean_s+std_s:.0f})")
axes[1, 0].axvline(mean_s + std_s, color="green", linestyle=":")
axes[1, 0].set_title("회차별 번호 6개 총합 분포 (정규분포 곡선)", fontsize=14, fontweight="bold")
axes[1, 0].set_xlabel("번호 총합")
axes[1, 0].legend()

# (4) 구간별(5개 색상 대역) 출현 비율
sec_labels = ["1~10 (노랑)", "11~20 (파랑)", "21~30 (빨강)", "31~40 (회색)", "41~45 (초록)"]
sec_counts = [
    count_series.loc[1:10].sum(),
    count_series.loc[11:20].sum(),
    count_series.loc[21:30].sum(),
    count_series.loc[31:40].sum(),
    count_series.loc[41:45].sum()
]
pie_colors = ["#fbc400", "#2b8be8", "#e74c3c", "#8e959d", "#27ae60"]
axes[1, 1].pie(sec_counts, labels=sec_labels, autopct="%1.1f%%", colors=pie_colors, startangle=140)
axes[1, 1].set_title("5개 번호대 구간별 출현 비율", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()


## 4. 스마트 AI 번호 조합 생성 엔진 (5가지 모드)
- **전략 1**: `golden` - 황금 밸런스 AI (가중치 + AC값 + 연속수/끝수/합계 필터링)
- **전략 2**: `cold` - 장기 미출현(콜드수) 반등형 (오래 안 나온 번호 2~3개 + 핫수 조화)
- **전략 3**: `hot` - 최근 대세(핫수) 추종형 (최근 10회차 다출수 중심)
- **전략 4**: `fixed_nums` - 고정수(반자동) 지정 모드 (생일 등 원하는 번호 1~3개 고정)
- **전략 5**: `exclude_nums` - 제외수 필터 모드 (안 나올 것 같은 번호 배제)

In [ ]:
# ==========================================
# 4. 스마트 번호 생성기 및 필터 함수
# ==========================================
past_winners = set([tuple(sorted(x)) for x in lotto_nums.values])
total_occurs = count_series.sum()
weights = [count_series.get(i, 0) / total_occurs for i in range(1, 46)]

def calculate_ac(nums):
    diffs = set(abs(a - b) for a, b in combinations(nums, 2))
    return len(diffs) - (len(nums) - 1)

def validate_combination(nums):
    nums = sorted(nums)
    if tuple(nums) in past_winners: return False, "과거 1등 중복"
    s = sum(nums)
    if not (100 <= s <= 200): return False, f"총합({s}) 범위 이탈"
    odds = sum(1 for n in nums if n % 2 != 0)
    if odds == 0 or odds == 6: return False, "극단적 홀짝"
    lows = sum(1 for n in nums if n <= 22)
    if lows == 0 or lows == 6: return False, "극단적 고저"
    for i in range(len(nums) - 2):
        if nums[i] + 1 == nums[i+1] and nums[i+1] + 1 == nums[i+2]:
            return False, "3연속 번호 포함"
    tails = Counter([n % 10 for n in nums])
    if max(tails.values()) >= 3: return False, "동일 끝수 3개 이상"
    if calculate_ac(nums) < 7: return False, "AC값 7 미만(단순 패턴)"
    return True, "통과"

def generate_smart_numbers(n_games=5, strategy="golden", fixed_nums=None, exclude_nums=None):
    fixed_nums = set(fixed_nums or [])
    exclude_nums = set(exclude_nums or [])
    pool = [n for n in range(1, 46) if n not in exclude_nums]
    
    recent_nums = []
    for _, r in df.tail(10).iterrows():
        recent_nums.extend(r[lotto_cols])
    hot_pool = [n for n, _ in Counter(recent_nums).most_common(12) if n in pool and n not in fixed_nums]
    cold_pool = [n for n, _ in cold_ranking[:12] if n in pool and n not in fixed_nums]
    
    results = []
    attempts = 0
    while len(results) < n_games and attempts < 2000:
        attempts += 1
        chosen = set(fixed_nums)
        need = 6 - len(chosen)
        
        if strategy == "cold" and len(cold_pool) >= 2:
            c_pick = random.sample(cold_pool, min(2, need))
            chosen.update(c_pick)
            rem = [n for n in pool if n not in chosen]
            if 6 - len(chosen) > 0 and len(rem) >= (6 - len(chosen)):
                chosen.update(random.sample(rem, 6 - len(chosen)))
        elif strategy == "hot" and len(hot_pool) >= 3:
            h_pick = random.sample(hot_pool, min(3, need))
            chosen.update(h_pick)
            rem = [n for n in pool if n not in chosen]
            if 6 - len(chosen) > 0 and len(rem) >= (6 - len(chosen)):
                chosen.update(random.sample(rem, 6 - len(chosen)))
        else:
            rem = [n for n in pool if n not in chosen]
            p = [weights[n-1] for n in rem]
            sum_p = sum(p)
            p = [x / sum_p for x in p]
            chosen.update(np.random.choice(rem, size=need, replace=False, p=p))
            
        candidate = sorted(list(chosen))
        if len(candidate) == 6:
            ok, _ = validate_combination(candidate)
            if ok and candidate not in [r["번호"] for r in results]:
                odds = sum(1 for n in candidate if n % 2 != 0)
                results.append({
                    "게임": f"게임 {len(results)+1}",
                    "번호": candidate,
                    "총합": sum(candidate),
                    "홀:짝": f"{odds}:{6-odds}",
                    "AC값": calculate_ac(candidate),
                    "전략": strategy
                })
    return pd.DataFrame(results)

def render_lotto_html(df_res):
    def ball_html(num):
        if num <= 10: c = "#fbc400"
        elif num <= 20: c = "#2b8be8"
        elif num <= 30: c = "#e74c3c"
        elif num <= 40: c = "#8e959d"
        else: c = "#27ae60"
        return f'<span style="display:inline-block;width:32px;height:32px;line-height:32px;border-radius:50%;background:{c};color:white;font-weight:bold;text-align:center;margin:2px 3px;box-shadow:0 2px 4px rgba(0,0,0,0.25);">{num}</span>'
    
    view = df_res.copy()
    view["번호"] = view["번호"].apply(lambda lst: "".join(ball_html(n) for n in lst))
    return HTML(view.to_html(escape=False, index=False, justify="center"))

# 번호 생성 예시 (5게임)
res = generate_smart_numbers(n_games=5, strategy="golden")
display(HTML("<h3>🌟 AI 황금 밸런스 추천 조합 (5게임)</h3>"))
display(render_lotto_html(res))


## 5. 과거 당첨 백테스팅 시뮬레이터 (내 번호 검증)
지정한 6개 번호가 역대 전체 회차(1회~최신) 동안 매주 샀다면 몇 번 당첨되었을지 시뮬레이션합니다.

In [ ]:
# ==========================================
# 5. 역대 당첨 백테스트 시뮬레이터
# ==========================================
def backtest(my_numbers):
    my_set = set(my_numbers)
    counts = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
    total_prize = 0
    wins = []
    
    for _, r in df.iterrows():
        draw = set(r[lotto_cols])
        bonus = r.get("보너스", r.get("보너스번호", 0))
        m = len(my_set.intersection(draw))
        rank = 0
        prize = 0
        if m == 6:
            rank = 1
            prize = r.get("1등당첨금액", 2000000000)
        elif m == 5 and bonus in my_set:
            rank = 2
            prize = 55000000
        elif m == 5:
            rank = 3
            prize = 1500000
        elif m == 4:
            rank = 4
            prize = 50000
        elif m == 3:
            rank = 5
            prize = 5000
            
        if rank > 0:
            counts[rank] += 1
            total_prize += prize
            wins.append((r["회차"], r["추첨일"], rank, m, prize))
            
    total_cost = len(df) * 1000
    roi = ((total_prize - total_cost) / total_cost) * 100
    
    print(f"🔍 [번호 검증] {sorted(list(my_numbers))}")
    print(f" - 분석 대상 회차: 1회 ~ {len(df)}회 (총 {len(df):,}주)")
    print(f" - 총 구매 비용: {total_cost:,}원 (매주 1천원 구매 기준)")
    print(f" - 총 당첨 금액: {total_prize:,}원 (수익률: {roi:+.1f}%)")
    print("-"*35)
    for rk in range(1, 6):
        print(f"   {rk}등: {counts[rk]}회 당첨")
    if wins:
        print(f"   최근 당첨 이력 (최대 5건): {wins[-5:]}")

# 예시: 추천 번호 중 1번 조합 백테스팅
test_numbers = res.iloc[0]["번호"]
backtest(test_numbers)
